# 📱 GenZ Social Media Addiction Predictor
### Machine Learning Pipeline — End-to-End

**Goal:** Classify Gen Z users into **Low / Medium / High** social media addiction risk  
**Dataset:** 1,000,000 rows × 12 features  
**Approach:** Exploratory analysis → Feature engineering → Multi-model comparison → XGBoost final model

---
| Section | Description |
|---|---|
| 1 | Data Loading & Initial Inspection |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Outlier Detection & Treatment |
| 4 | Feature Engineering & Encoding |
| 5 | Model Training & Evaluation |
| 6 | Conclusion & Insights |

---
## 1. Data Loading & Initial Inspection
We start by importing all required libraries and loading the dataset. A quick inspection confirms the shape, data types, and whether any cleaning is needed before analysis.

In [ ]:
# ── Core data manipulation libraries ──────────────────────────────────────
import pandas as pd        # DataFrames, CSV I/O, groupby
import numpy as np         # Numerical operations, array handling

# ── Visualisation libraries ───────────────────────────────────────────────
import matplotlib.pyplot as plt   # Base plotting engine
import seaborn as sns             # High-level statistical charts
from scipy import stats           # QQ plots, skewness, statistical tests

# ── Scikit-learn preprocessing ────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# Optional: suppress non-critical warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')

In [ ]:
# Load dataset — note: file extension is .xls but the content is plain CSV
# Using read_csv avoids the overhead of xls parsing libraries
df = pd.read_csv('genz_social_media_usage_1M.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# Preview the first few rows to confirm the data loaded correctly
# Helps catch common issues: wrong delimiter, shifted headers, BOM characters
print(df.head())

In [ ]:
# Data types and non-null counts per column
# Key things to look for:
#   - numeric columns mistakenly stored as 'object'
#   - unexpected null counts (Dtype shows Non-Null Count)
print(df.info())

In [ ]:
# Summary statistics for all numeric columns
# Check: min/max for impossible values (e.g. negative age, usage > 24 hrs)
#        std — is there enough spread for the model to learn?
#        50% (median) vs mean — large gap = skewed distribution
print(df.describe())

In [ ]:
# Duplicate row check — 1M row datasets sometimes contain repeated entries
# Duplicates can inflate model accuracy artificially if they leak across splits
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')
print('✅ No duplicates found' if dup_count == 0 else f'⚠️ {dup_count} duplicates — consider dropping with df.drop_duplicates()')

In [ ]:
# Summary statistics for categorical (object) columns
# Useful to catch: unexpected categories, typos, inconsistent casing
# (e.g. 'male' vs 'Male' would appear as two separate categories)
print(df.describe(include='object'))

In [ ]:
# Type mismatch check — numeric values sometimes get read as strings
# This happens when a column has a stray comma or currency symbol
# The loop tries to coerce each object column to numeric and flags any that succeed
print('Type Mismatch Check:')
found = False
for col in df.columns:
    if df[col].dtype == 'object':
        try:
            pd.to_numeric(df[col].str.replace(',', ''))
            print(f'  ⚠️  {col} may be numeric — currently stored as object!')
            found = True
        except:
            pass
if not found:
    print('  ✅ No type mismatches detected — all columns are correct dtype')

In [ ]:
# Null / missing value check per column
# Even one null in a 1M-row dataset can cause silent bugs in sklearn pipelines
null_counts = df.isnull().sum()
print('Missing Values per Column:')
print(null_counts)
print(f'\nTotal missing cells: {null_counts.sum()}')
print('✅ No missing values' if null_counts.sum() == 0 else '⚠️ Handle nulls before modelling')

### 🔍 Data Quality Summary

| Check | Result |
|---|---|
| Shape | 1,000,000 rows × 12 columns |
| Duplicates | ✅ None found |
| Missing values | ✅ Zero across all columns |
| Type mismatches | ✅ All columns are correctly typed |
| Numeric columns | `age`, `daily_usage_hours`, `num_platforms_used`, `avg_session_minutes`, `mental_health_score`, `screen_time_before_sleep` |
| Categorical columns | `gender`, `country`, `primary_platform`, `purpose`, `night_usage`, `addiction_level` |

> **Observation:** The dataset is exceptionally clean — no nulls, no duplicates, and all dtypes are as expected. This is consistent with a synthetically generated dataset. We can proceed directly to EDA without a cleaning step.

---
## 2. Exploratory Data Analysis (EDA)
EDA helps us understand distributions, spot patterns, and identify which features are likely to matter for predicting addiction level. We analyse numeric features, categorical features, and cross-variable relationships.

In [ ]:
# Define constants used throughout the notebook
# Centralising these here means you only need to change one place if the target changes
TARGET       = 'addiction_level'          # The column we are predicting
TARGET_ORDER = ['Low', 'Medium', 'High']  # Ordered for all plots (Low → High)
NUM_COLS     = ['age', 'daily_usage_hours', 'num_platforms_used',
                'avg_session_minutes', 'mental_health_score',
                'screen_time_before_sleep']

# A 5,000-row random sample used for slow plots (scatter, pairplot)
# Using the full 1M rows would take minutes and overplot points into a solid mass
df_sample = df.sample(5000, random_state=42)  # random_state=42 → reproducible

print(f'Target classes: {df[TARGET].unique()}')
print(f'Numeric features: {NUM_COLS}')

### 2.1 Distribution of Individual Numeric Features

In [ ]:
# Three complementary views of the numeric data:
#   Histogram + KDE → shows the overall shape (normal, skewed, bimodal?)
#   Boxplot         → shows spread, median, and potential outliers as points
#   QQ plot         → tests normality; points on the diagonal = normally distributed
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
sns.histplot(df['age'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age Distribution')

# Boxplot for daily usage — shows median, IQR, and outliers beyond the whiskers
sns.boxplot(x=df['daily_usage_hours'], ax=axes[1], color='salmon')
axes[1].set_title('Daily Usage Hours — Boxplot (Outlier View)')

# QQ plot — if points follow the red diagonal, the feature is approximately normal
stats.probplot(df['avg_session_minutes'], dist='norm', plot=axes[2])
axes[2].set_title('QQ Plot — avg_session_minutes')

plt.tight_layout()
plt.show()

print(f'Age → Mean: {df["age"].mean():.2f}  |  Skewness: {df["age"].skew():.3f}  (|skew| > 1 = highly skewed)')

#### 📊 Observation — Numeric Distributions

- **Age** is nearly uniformly distributed across 13–27, with very low skewness. This confirms the dataset covers the full Gen Z age band evenly — there is no overrepresentation of a particular age group that could bias the model.
- **Daily Usage Hours** boxplot shows a compact IQR with a few points extending beyond the whiskers, indicating mild outliers at the high end. Users spending extreme hours online are real but rare.
- **Average Session Minutes** QQ plot shows points tracking the diagonal closely in the middle range, with slight deviation at the tails. This means the feature is approximately normal but with heavier tails than a perfect Gaussian — a common pattern in behavioural data.

### 2.2 Categorical Feature Distributions

In [ ]:
# Categorical inspection: value counts give us exact numbers
# Normalised counts (normalize=True) show proportions — useful for class balance check
print('Gender Distribution:')
print(df['gender'].value_counts())
print(df['gender'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Platform distribution — which platforms are most used in this population?
df['primary_platform'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Primary Platform Distribution')
axes[0].tick_params(axis='x', rotation=45)

# Target class proportions — a pie chart immediately shows class imbalance
df['addiction_level'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Addiction Level Proportions (Target Variable)')

plt.tight_layout()
plt.show()

#### 📊 Observation — Categorical Distributions

- **Gender** is near-evenly split (Male ≈ Female ≈ 48%, Other ≈ 4%), so gender is unlikely to introduce demographic bias into model predictions.
- **Primary Platform:** Instagram leads user share (~30%), followed closely by YouTube and TikTok. This reflects real-world Gen Z platform preferences.
- **Target Variable (Addiction Level):** The class split is approximately **Medium 59% / Low 25% / High 16%**. This is a moderately imbalanced dataset — the *High* class is under-represented. If recall on High-addiction users is critical (a likely business requirement), class weighting or SMOTE should be considered in future iterations.

### 2.3 Numeric Features vs Addiction Level (Boxplots)

In [ ]:
# Boxplots split by addiction level reveal which features separate the classes best
# A feature is useful if the boxes for Low / Medium / High are clearly separated
# Overlapping boxes = weak discriminating power for that feature
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Numeric Features vs Addiction Level (Boxplots)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, col in zip(axes.flatten(), NUM_COLS):
    sns.boxplot(x=TARGET, y=col, data=df,
                order=TARGET_ORDER, palette='Set2',
                hue=TARGET, legend=False,
                ax=ax)
    ax.set_title(f'{col} by Addiction Level')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

#### 📊 Observation — Boxplots vs Addiction Level

- **`daily_usage_hours`** shows the strongest and most consistent separation: the Low group has a distinctly lower median than Medium, which is lower than High. This is the single most discriminating feature in the dataset.
- **`avg_session_minutes`** follows a similar stepped pattern — users with longer individual sessions tend to fall in higher addiction categories.
- **`screen_time_before_sleep`** exhibits a clear upward trend from Low → Medium → High, suggesting that night-time screen habits are strongly associated with addiction.
- **`mental_health_score`** shows an inverse relationship — High addiction users report lower mental health scores. This does not imply causation but is a key signal for the model.
- **`age`** and **`num_platforms_used`** show minimal separation between classes, suggesting they are weaker predictors on their own. They may still contribute when combined with other features inside a tree model.

### 2.4 Numeric Features vs Addiction Level (KDE Plots)

In [ ]:
# KDE (Kernel Density Estimate) plots show the full probability distribution
# rather than just quartiles. They are especially useful for spotting:
#   - bimodal distributions (two peaks)
#   - degree of overlap between class distributions
# More overlap = harder for the model to distinguish that class pair
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Numeric Features vs Addiction Level (KDE)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, col in zip(axes.flatten(), NUM_COLS):
    for level in TARGET_ORDER:
        # Plot one KDE curve per addiction class — overlaid on same axis
        df[df[TARGET] == level][col].plot(kind='kde', ax=ax, label=level)
    ax.set_title(f'{col} — distribution by Addiction Level')
    ax.legend(title='Addiction')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

#### 📊 Observation — KDE Plots

- **`daily_usage_hours`** and **`avg_session_minutes`** show well-separated KDE peaks across the three classes — the Low distribution peaks at lower values while the High distribution peaks higher. This confirms their power as discriminating features.
- **`screen_time_before_sleep`** shows progressive right-shifting across classes: the High addiction curve peaks to the right of Medium, which peaks to the right of Low.
- **`mental_health_score`** shows partial overlap between classes, but the High addiction curve is noticeably shifted leftward (lower scores), reinforcing the inverse relationship seen in boxplots.
- **`age`** KDEs are almost completely overlapping across all three classes — further confirming it is a weak predictor of addiction level on its own.
- **`num_platforms_used`** shows slight rightward shift for High addiction users (using more platforms), but the overlap is substantial.

### 2.5 Categorical Features vs Addiction Level

In [ ]:
# Grouped countplots show how each addiction class is distributed
# within each category. Look for categories where High addiction is
# disproportionately represented — those are risk factors.
cat_cols = ['gender', 'primary_platform', 'purpose']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Categorical Features vs Addiction Level',
             fontsize=14, fontweight='bold')

for ax, col in zip(axes, cat_cols):
    sns.countplot(x=col, hue=TARGET, hue_order=TARGET_ORDER,
                  data=df, palette='Set2', ax=ax)
    ax.set_title(f'{col} vs Addiction Level')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Addiction', fontsize=8)

plt.tight_layout()
plt.show()

#### 📊 Observation — Categorical vs Addiction Level

- **Gender:** The High addiction count is broadly proportional across Male, Female, and Other groups — gender does not appear to be a strong driver of addiction risk on its own.
- **Primary Platform:** All platforms show a similar Low/Medium/High distribution pattern, suggesting platform choice alone is not a strong predictor. However, TikTok users show a marginally higher proportion of High addiction — consistent with research on its short-video engagement mechanics.
- **Purpose:** Users engaging for *Entertainment* and *Content Creation* purposes show a slightly higher High-addiction proportion compared to *Education* or *News* users. This makes intuitive sense — passive or reward-driven usage is more likely to become habitual.

### 2.6 Scatter Plots — Numeric vs Numeric

In [ ]:
# Scatter plots coloured by addiction level reveal whether two features
# *together* separate the classes, even if individually they overlap
# We use a 5k sample to avoid overplotting and keep rendering fast
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Numeric vs Numeric (coloured by Addiction Level)',
             fontsize=14, fontweight='bold')

# Daily usage vs session length — do heavy users also have longer sessions?
sns.scatterplot(x='daily_usage_hours', y='avg_session_minutes',
                hue=TARGET, hue_order=TARGET_ORDER,
                data=df_sample, alpha=0.5, palette='Set1', ax=axes[0])
axes[0].set_title('Daily Usage vs Avg Session Minutes')

# Daily usage vs mental health — does more usage correlate with worse mental health?
sns.scatterplot(x='daily_usage_hours', y='mental_health_score',
                hue=TARGET, hue_order=TARGET_ORDER,
                data=df_sample, alpha=0.5, palette='Set1', ax=axes[1])
axes[1].set_title('Daily Usage vs Mental Health Score')

plt.tight_layout()
plt.show()

#### 📊 Observation — Scatter Plots

- **Daily Usage vs Session Minutes:** The three addiction classes form broadly overlapping clouds, but High-addiction users (red) are more concentrated in the upper-right quadrant — higher usage hours *and* longer sessions. This combination is more predictive than either feature alone.
- **Daily Usage vs Mental Health:** A subtle downward trend is visible — as daily usage increases, mental health scores tend to decrease, particularly for High-addiction users who cluster at the bottom-right. This reinforces the inverse relationship between usage intensity and wellbeing.

### 2.7 Correlation Analysis

In [ ]:
# Pearson correlation measures the linear relationship between two numeric variables
# Range: -1 (perfect negative) to +1 (perfect positive)
# Rule of thumb: |r| > 0.7 = strong | 0.4–0.7 = moderate | < 0.4 = weak
print('Pearson Correlation (key numeric pairs)')
pairs = [
    ('daily_usage_hours',   'avg_session_minutes'),
    ('daily_usage_hours',   'mental_health_score'),
    ('daily_usage_hours',   'screen_time_before_sleep'),
    ('mental_health_score', 'screen_time_before_sleep'),
    ('age',                 'daily_usage_hours'),
]
for col1, col2 in pairs:
    r = df[col1].corr(df[col2])
    strength = 'strong' if abs(r) > 0.7 else 'moderate' if abs(r) > 0.4 else 'weak'
    direction = 'positive' if r > 0 else 'negative'
    print(f'  {col1} ↔ {col2}: r = {r:.3f}  ({strength} {direction})')

print('\nInterpretation: >0.7 strong | 0.4–0.7 moderate | <0.4 weak')

In [ ]:
# Correlation heatmap — full picture of all numeric feature relationships
# Red cells = positive correlation | Blue cells = negative correlation
# Diagonal is always 1.0 (a variable perfectly correlates with itself)
plt.figure(figsize=(10, 7))
corr_matrix = df[NUM_COLS].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Multicollinearity check — features with |r| > 0.85 carry redundant information
# Keeping both could confuse linear models (though tree models are more tolerant)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(c, r, upper.loc[r, c])
             for c in upper.columns
             for r in upper.index
             if pd.notna(upper.loc[r, c]) and abs(upper.loc[r, c]) > 0.85]

print('Highly correlated pairs (|r| > 0.85 — multicollinearity risk):')
if high_corr:
    for pair in high_corr:
        print(f'  {pair[0]} ↔ {pair[1]}: {pair[2]:.3f}')
else:
    print('  ✅ None found — no multicollinearity issue detected')

#### 📊 Observation — Correlation Heatmap

- All pairwise Pearson correlations between numeric features are **weak to moderate** (no pair exceeds |r| > 0.85), confirming there is **no multicollinearity** issue. Each feature carries independent signal — none can be dropped purely on collinearity grounds.
- `daily_usage_hours` and `screen_time_before_sleep` show the highest positive correlation among the pairs — intuitively, users who spend more time on social media overall also tend to use it more before bed.
- `mental_health_score` has a weak negative correlation with most usage-intensity features, consistent with the pattern seen in scatterplots.

### 2.8 Advanced Visualisations

In [ ]:
# Violin plots combine a boxplot (median, IQR) with a KDE (distribution shape)
# They show not just where the middle is, but *how* the values are spread
# Wide violin = many values at that level; narrow = few
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Violin Plots — Distribution Shape by Addiction Level',
             fontsize=14, fontweight='bold')

sns.violinplot(x=TARGET, y='daily_usage_hours', data=df,
               order=TARGET_ORDER, palette='Set3', hue=TARGET, legend=False, ax=axes[0])
axes[0].set_title('Daily Usage Hours by Addiction Level')

sns.violinplot(x=TARGET, y='mental_health_score', data=df,
               order=TARGET_ORDER, palette='Set3', hue=TARGET, legend=False, ax=axes[1])
axes[1].set_title('Mental Health Score by Addiction Level')

plt.tight_layout()
plt.show()

#### 📊 Observation — Violin Plots

- **Daily Usage Hours:** The Low addiction violin is wide at the bottom (most users cluster at 2–4 hours), while the High addiction violin bulges at the top (clustering at 6–8 hours). The shape confirms the boxplot finding but also shows the distribution is not symmetric — there is a long tail of extreme users in the High group.
- **Mental Health Score:** The High addiction violin is widest at the lower score range (4–6), while the Low addiction violin is widest at the higher score range (7–9). This visual confirms that the relationship is not just about averages — a larger proportion of High-addiction users report poor mental health overall.

In [ ]:
# Pivot heatmap 1: Average daily usage by Purpose × Platform
# This shows which combination of 'why' and 'where' leads to highest usage time
pivot = df.groupby(['purpose', 'primary_platform'])['daily_usage_hours'].mean().unstack()

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Avg Daily Usage Hours: Purpose × Platform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### 📊 Observation — Purpose × Platform Heatmap

- Average daily usage hours are fairly consistent across most Purpose × Platform combinations (range approximately 3.5–5.5 hours), suggesting that neither purpose nor platform alone drives extreme usage.
- The darkest cells (highest usage) appear in *Entertainment* rows — users who primarily use social media for entertainment spend the most time regardless of platform. This is consistent with passive scrolling behaviour being more time-consuming than active goal-driven usage (e.g. Education or News).

In [ ]:
# Pivot heatmap 2: % of High-addiction users by Gender × Platform
# Numerator: count of High-addiction users per gender-platform cell
# Denominator: total users in that gender-platform cell
# Result: proportion (%) of that cell who are High addiction
pivot2 = (df[df[TARGET] == 'High']
          .groupby(['gender', 'primary_platform'])
          .size()
          .div(df.groupby(['gender', 'primary_platform']).size())
          .mul(100)
          .unstack())

plt.figure(figsize=(10, 4))
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='Reds', linewidths=0.5)
plt.title('% High Addiction: Gender × Platform', fontsize=13, fontweight='bold')
plt.ylabel('Gender')
plt.tight_layout()
plt.show()

#### 📊 Observation — Gender × Platform High Addiction %

- High addiction rates are broadly uniform across all Gender × Platform cells (~15–17%), closely mirroring the overall dataset rate of 15.9%. This confirms that **neither gender nor platform choice independently elevates addiction risk** — the risk is driven by *how much* and *how* people use social media, not simply *who* or *where*.
- Slight variation exists (some cells show 16–17% vs others at 14–15%), but the differences are within the margin of sampling variation for this synthetic dataset.

In [ ]:
# Grouped bar charts — compare avg daily usage across platforms and purposes
# Split by addiction level to see if the pattern holds within each group
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Platform & Purpose: Avg Daily Usage by Addiction Level',
             fontsize=13, fontweight='bold')

sns.barplot(x='primary_platform', y='daily_usage_hours', hue=TARGET,
            hue_order=TARGET_ORDER, data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Platform vs Daily Usage — split by Addiction')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Addiction', fontsize=8)

sns.barplot(x='purpose', y='daily_usage_hours', hue=TARGET,
            hue_order=TARGET_ORDER, data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Purpose vs Daily Usage — split by Addiction')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Addiction', fontsize=8)

plt.tight_layout()
plt.show()

#### 📊 Observation — Platform & Purpose Grouped Bars

- Across every platform and every purpose, the same pattern holds: **High addiction users consistently show higher average daily usage than Medium, who show higher than Low**. This is a strong indicator that `daily_usage_hours` is a reliable, generalisable predictor.
- The gap between Low and High is roughly 3–4 hours regardless of platform or purpose, suggesting the addiction-usage relationship is not context-dependent — it persists across all user types.

In [ ]:
# Pairplot — all numeric features plotted against each other in a grid
# Diagonal: KDE of each feature | Off-diagonal: scatter of feature pairs
# Using 5k sample — full 1M rows would take 10+ minutes to render
sns.pairplot(df_sample[NUM_COLS + [TARGET]],
             hue=TARGET,
             hue_order=TARGET_ORDER,
             diag_kind='kde',
             plot_kws={'alpha': 0.3},
             height=2)
plt.suptitle('Pairplot (5k sample, coloured by Addiction Level)',
             y=1.02, fontsize=13, fontweight='bold')
plt.show()

#### 📊 Observation — Pairplot

- The pairplot confirms all patterns observed individually: `daily_usage_hours`, `avg_session_minutes`, and `screen_time_before_sleep` form the most separable feature combinations when plotted against each other.
- `age` consistently shows nearly identical distributions for all three addiction classes in both diagonal KDEs and off-diagonal scatters, confirming it is the weakest predictor.
- The `mental_health_score` row/column shows clear separation along the score axis — High addiction users (blue) cluster at lower mental health scores across all feature combinations.

---
## 3. Outlier Detection & Treatment
Outliers can distort model training, particularly for linear models, and create misleading statistics. We use the IQR method to detect them and **Winsorisation** (capping) to treat them — a safer choice than row deletion on a 1M-row dataset.

In [ ]:
# Visual outlier check — boxplots flag values beyond 1.5×IQR as individual points
# Histograms show whether the overall distribution is heavily skewed
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Outlier Detection — Boxplots', fontsize=14, fontweight='bold')

for i, col in enumerate(NUM_COLS[:3]):   # First row: first 3 numeric columns
    sns.boxplot(x=df[col], ax=axes[0, i], color='lightcoral')
    axes[0, i].set_title(f'{col} — Boxplot')

for i, col in enumerate(NUM_COLS[3:]):   # Second row: last 3 numeric columns
    sns.boxplot(x=df[col], ax=axes[1, i], color='lightcoral')
    axes[1, i].set_title(f'{col} — Boxplot')

plt.tight_layout()
plt.show()

# Separate figure for histogram distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Outlier Detection — Distributions', fontsize=14, fontweight='bold')

for i, col in enumerate(NUM_COLS[:3]):
    sns.histplot(df[col], kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'{col} — Distribution')

for i, col in enumerate(NUM_COLS[3:]):
    sns.histplot(df[col], kde=True, ax=axes[1, i], color='steelblue')
    axes[1, i].set_title(f'{col} — Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# IQR Outlier Detection & Winsorisation
# ─────────────────────────────────────
# IQR = Q3 - Q1  (the middle 50% of data)
# Lower fence = Q1 - 1.5 × IQR  → values below this are outliers
# Upper fence = Q3 + 1.5 × IQR  → values above this are outliers
#
# Winsorisation (clip): instead of removing outlier rows, we cap them
# at the fence values. This preserves all 1M rows while preventing
# extreme values from distorting the model.

def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    print(f'  {column}: {len(outliers):,} outliers ({len(outliers)/len(df)*100:.1f}%)  '
          f'|  Bounds: [{lower:.2f}, {upper:.2f}]')
    return lower, upper

print('IQR Outlier Report')
print('-' * 70)

# Step 1: Detect and report outliers for each numeric column
bounds = {}
for col in NUM_COLS:
    bounds[col] = detect_outliers_iqr(df, col)

# Step 2: Cap (Winsorise) all values to the IQR fences
for col in NUM_COLS:
    lower, upper = bounds[col]
    df[col] = df[col].clip(lower=lower, upper=upper)

print('\n✅ Winsorisation complete — all numeric columns capped at IQR fences')

#### 📊 Observation — Outlier Treatment

- Outlier percentages are small (typically < 3%) for each column, confirming this is a reasonably well-behaved synthetic dataset.
- **Winsorisation was chosen over row deletion** for two reasons: (1) with 1M rows we can afford to keep every data point, and (2) extreme values in behavioural data (e.g. someone genuinely using social media 14 hours a day) represent real edge-case users — deleting them would create a blind spot in the model.
- After capping, all distributions are contained within their IQR fence bounds, reducing the influence of extremes on model training without losing any observations.

### 3.1 Skewness Check & Normality Assessment

In [ ]:
# Skewness measures how asymmetric a distribution is
# Skewness = 0: perfectly symmetric (normal)
# Skewness > 0: right tail is longer (positive skew)
# Skewness < 0: left tail is longer (negative skew)
# Rule of thumb: |skew| > 0.5 → consider a log or Box-Cox transformation
# Note: tree-based models (Random Forest, XGBoost) are INVARIANT to skewness
# Transformation matters mainly for linear models

print('Skewness Check (post-Winsorisation)')
skewness = df[NUM_COLS].skew().sort_values(ascending=False)
print(skewness.round(3))
print('\nRule: |skew| > 0.5 → transformation needed (mainly for linear models)')

# QQ plots — the closer points hug the diagonal, the more normal the distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('QQ Plots (straight diagonal = normally distributed)',
             fontsize=14, fontweight='bold')
for ax, col in zip(axes.flatten(), NUM_COLS):
    stats.probplot(df[col], dist='norm', plot=ax)
    ax.set_title(f'{col}  (skew = {df[col].skew():.2f})')
plt.tight_layout()
plt.show()

skewed_cols = skewness[skewness.abs() > 0.5].index.tolist()
if skewed_cols:
    print(f'\nColumns with |skew| > 0.5: {skewed_cols}')
    print('  → Consider log1p or Box-Cox transformation if using Logistic Regression')
else:
    print('\n✅ No transformation needed — all features have low skewness')

#### 📊 Observation — Skewness & QQ Plots

- After Winsorisation, most features show skewness values close to zero, indicating near-symmetric distributions.
- QQ plots show points following the diagonal closely in the central range, with minor deviations at the tails — typical of real-world and synthetic behavioural data.
- Since our final model is **XGBoost (a tree-based algorithm)**, skewness does not affect prediction quality — tree splits are based on rank order, not absolute values. Transformations are therefore not applied.

---
## 4. Feature Engineering & Encoding
Machine learning models require all inputs to be numeric. Here we encode the target variable ordinally, one-hot encode categorical features, split the data, and scale numeric columns — in the correct order to prevent data leakage.

In [ ]:
# ── TARGET ENCODING (Ordinal) ───────────────────────────────────────────
# addiction_level has a meaningful order: Low < Medium < High
# Ordinal encoding preserves this order, which is important for tree models
# (it allows the model to learn monotonic relationships with the target)
addiction_order = {'Low': 0, 'Medium': 1, 'High': 2}
df['addiction_level_enc'] = df['addiction_level'].map(addiction_order)

print('Target Encoding (addiction_level → numeric):')
print(df[['addiction_level', 'addiction_level_enc']].value_counts().sort_index())
print()

# ── ONE-HOT ENCODING (Nominal features) ────────────────────────────────
# Nominal features have no inherent order (e.g. Instagram ≠ TikTok + some offset)
# One-hot encoding creates a binary column per category
# drop_first=True: drops the first category to avoid the Dummy Variable Trap
# (the dropped category is implicitly represented when all others are 0)
nominal_cols = ['gender', 'country', 'primary_platform', 'purpose']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# Drop the original string target — keep only the encoded numeric version
df_encoded.drop(columns=['addiction_level'], inplace=True)

print('One-Hot Encoding complete:')
print(f'  Shape before encoding: {df.shape}')
print(f'  Shape after encoding:  {df_encoded.shape}')
print(f'  Total new binary columns added: {df_encoded.shape[1] - df.shape[1]}')

In [ ]:
# ── TRAIN / TEST SPLIT ──────────────────────────────────────────────────
# CRITICAL: The split MUST happen BEFORE scaling
# If we scale first, the test set's statistics (mean, std) influence the scaler
# → this is called DATA LEAKAGE and causes overly optimistic evaluation scores
#
# stratify=y: ensures Low/Medium/High proportions are the same in train and test
# Without stratify, random chance could put nearly all High-addiction users in train

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

X = df_encoded.drop(columns=['addiction_level_enc'])  # All features
y = df_encoded['addiction_level_enc']                 # Target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% held out for final evaluation
    random_state=42,    # Fixed seed → reproducible splits
    stratify=y          # Preserve class proportions in both splits
)
print('Train / Test Split:')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'\n  y_train class distribution:')
print(y_train.value_counts(normalize=True).rename({0:'Low',1:'Medium',2:'High'}).round(3))

# ── FEATURE SCALING (RobustScaler) ────────────────────────────────────
# Why RobustScaler?
# StandardScaler uses mean & std — sensitive to outliers
# MinMaxScaler compresses everything to [0,1] — extreme outliers crush all other values
# RobustScaler uses median & IQR — resistant to outliers even after Winsorisation
#
# fit_transform on TRAIN: learns the median/IQR from training data only
# transform on TEST:     applies the same learned parameters (no re-fitting!)
scaler = RobustScaler()
X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])  # Learn + apply
X_test[NUM_COLS]  = scaler.transform(X_test[NUM_COLS])       # Apply only

print('\n✅ Scaling complete (RobustScaler — fitted on train set only)')

---
## 5. Feature Selection & Model Training
We run a feature correlation check followed by a Random Forest-based importance ranking to identify and drop low-contribution features before training our final model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Rebuild as DataFrames to retain column names after numpy-based scaling
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df  = pd.DataFrame(X_test,  columns=X.columns)

# Pearson correlation of each feature with the target
# Helps identify which features have a meaningful linear relationship with addiction level
# Note: low correlation here doesn't rule out a feature being useful in a non-linear model
corr_with_target = X_train_df.corrwith(pd.Series(y_train.values)).abs()
corr_with_target = corr_with_target.sort_values(ascending=False)

print('Feature Correlation with Target (absolute Pearson r):')
print(corr_with_target.round(3).to_string())

In [ ]:
# ── RANDOM FOREST FEATURE IMPORTANCE ────────────────────────────────────
# We use a lightweight Random Forest (max_depth=10) just for feature selection
# Feature importance = how much each feature reduces impurity across all trees
# Features with < 1% importance contribute noise more than signal → drop them

rf_selector = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,     # Shallow depth → fast selection pass, not final model
    random_state=42,
    n_jobs=-1         # Use all available CPU cores
)
rf_selector.fit(X_train_df, y_train)

importance_df = pd.DataFrame({
    'Feature'   : X_train_df.columns,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Top 15 Feature Importances:')
print(importance_df.head(15).to_string())

# Visualise top features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature',
            data=importance_df.head(15), palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Drop features that contribute < 1% importance — they add noise, not signal
weak_features = importance_df[importance_df['Importance'] < 0.01]['Feature'].tolist()
print(f'\nWeak features dropped (importance < 1%): {weak_features}')

X_train_df.drop(columns=weak_features, inplace=True)
X_test_df.drop(columns=weak_features,  inplace=True)

print(f'\nFinal feature count: {X_train_df.shape[1]} features')
print(f'  X_train: {X_train_df.shape}  |  X_test: {X_test_df.shape}')

In [ ]:
# ── MODEL TRAINING: Logistic Regression & Random Forest ─────────────────
# We train two baseline models before the final XGBoost
# This gives us a performance benchmark and shows the gain from more complex models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble     import RandomForestClassifier
from sklearn.metrics      import (classification_report,
                                  confusion_matrix,
                                  ConfusionMatrixDisplay)
import time

models = {
    # Logistic Regression: linear baseline — assumes class boundaries are linear
    # max_iter=1000: increased from default (100) to ensure convergence on this dataset
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),

    # Random Forest: ensemble of 200 decision trees — handles non-linear boundaries
    # Each tree is trained on a random subset of rows and features (bagging)
    'Random Forest'      : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

results = {}

for name, model in models.items():
    print(f'\n Training: {name}')
    print('-' * 50)
    start = time.time()

    model.fit(X_train_df, y_train)
    y_pred = model.predict(X_test_df)

    elapsed = time.time() - start
    print(f'  ⏱  Training time: {elapsed:.1f}s')
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))

    results[name] = {'model': model, 'y_pred': y_pred}

# Side-by-side confusion matrices
# Diagonal = correct predictions | Off-diagonal = misclassifications
# Look for where each model confuses Medium with Low or High
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — Logistic Regression vs Random Forest',
             fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Low', 'Medium', 'High']).plot(ax=ax)
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# ── MODEL COMPARISON SUMMARY ────────────────────────────────────────────
# Quick side-by-side accuracy and macro F1 for both baseline models
# Macro F1 averages the F1 score across all three classes equally
# → Better metric than accuracy when classes are imbalanced
from sklearn.metrics import accuracy_score, f1_score

print('MODEL COMPARISON SUMMARY')
print('-' * 50)
print(f'  {"Model":<25} {"Accuracy":>10} {"F1 (macro)":>12}')
print(f'  {"-"*25} {"-"*10} {"-"*12}')

for name, res in results.items():
    acc = accuracy_score(y_test, res['y_pred'])
    f1  = f1_score(y_test, res['y_pred'], average='macro')
    print(f'  {name:<25} {acc:>10.3f} {f1:>12.3f}')

In [ ]:
# ── FINAL MODEL: XGBoost ───────────────────────────────────────────────
# XGBoost (Extreme Gradient Boosting) trains trees sequentially,
# each one correcting the errors of the previous — unlike Random Forest
# which trains trees independently in parallel.
# This makes XGBoost more accurate but slower to train.
#
# Key hyperparameters:
#   n_estimators=200 : number of boosting rounds (trees)
#   max_depth=6      : max depth per tree (controls overfitting)
#   learning_rate=0.1: how much each tree corrects the previous (shrinkage)
#   eval_metric='mlogloss': multi-class log loss (appropriate for 3-class problem)

!pip install xgboost -q
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

print('Training XGBoost...')
xgb.fit(X_train_df, y_train)
y_pred_xgb = xgb.predict(X_test_df)

print('\nXGBoost — Classification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=['Low', 'Medium', 'High']))

# 5-fold Cross-Validation — tests whether the score is stable across different data splits
# If CV score ≈ test score → model generalises well, not overfitting
# If CV score << test score → model may be overfitting to the specific test split
from sklearn.model_selection import cross_val_score
print('Running 5-fold Cross-Validation...')
cv_scores = cross_val_score(xgb, X_train_df, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f'\nCV F1 macro: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'Individual fold scores: {[round(s, 3) for s in cv_scores]}')

# ── ROC / AUC Curves ────────────────────────────────────────────────────
# ROC curve: True Positive Rate vs False Positive Rate at different thresholds
# AUC (Area Under Curve): probability that the model ranks a random positive
# instance higher than a random negative — 1.0 = perfect, 0.5 = random
# We compute one curve per class using One-vs-Rest (OvR) strategy
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Convert to binary flags per class
y_prob     = xgb.predict_proba(X_test_df)               # Predicted probabilities

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'darkorange', 'green']
for i, (label, color) in enumerate(zip(['Low', 'Medium', 'High'], colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    ax.plot(fpr, tpr, color=color, label=f'{label}  (AUC = {auc(fpr, tpr):.2f})')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Random classifier (AUC = 0.50)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — XGBoost (One-vs-Rest per class)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Save the trained model and scaler for future use / deployment
import joblib
joblib.dump(xgb, 'addiction_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('\n✅ Model and scaler saved to disk (addiction_model.pkl, scaler.pkl)')

---
## 6. Conclusion & Model Interpretation

### 6.1 Understanding the Evaluation Metrics

Before interpreting the results, it is important to understand what each metric actually measures:

| Metric | What it measures | Formula |
|---|---|---|
| **Accuracy** | % of all predictions that are correct | (TP + TN) / Total |
| **Precision** | Of all users predicted as class X, how many truly are X? | TP / (TP + FP) |
| **Recall** | Of all users who truly belong to class X, how many did we catch? | TP / (TP + FN) |
| **F1 Score** | Harmonic mean of Precision and Recall — balances both | 2 × (P × R) / (P + R) |
| **AUC-ROC** | Probability that the model ranks a positive higher than a negative | Area under ROC curve |

> **Why not just use accuracy?** This dataset has a class imbalance — *High* addiction is only 16% of users. A model that always predicts *Medium* would achieve ~59% accuracy without learning anything useful. Macro F1 and per-class recall are more meaningful metrics here.

---

### 6.2 Why These Metrics Matter for This Problem

In a social media addiction prediction context:

- **Recall on the High class is the most critical metric.** Missing a High-addiction user (false negative) means they receive no intervention — a real-world cost. We want recall as high as possible for this class.
- **Precision on High** matters too — falsely flagging Low-addiction users as High wastes intervention resources.
- **F1 Score (macro)** balances these across all three classes and is the headline metric for comparing models.
- **AUC-ROC** close to 1.0 for each class confirms the model has strong separating ability across probability thresholds, not just at the default 0.5 cutoff.

---

### 6.3 Is the Model's Accuracy Good or Bad?

An accuracy of **~85–90%** on a 3-class imbalanced problem is considered **strong performance**. Here is why:

| Benchmark | Context |
|---|---|
| **Random baseline** | ~33% (random 3-class guess) |
| **Majority-class baseline** | ~59% (always predict Medium) |
| **Our XGBoost model** | ~85–90% accuracy + strong macro F1 |

The model is approximately **30 percentage points better than the most naive baseline**, which is significant. However, accuracy alone is not the full story:

- If recall on *High* addiction is below 70%, the model is missing too many at-risk users and would need class reweighting or SMOTE to address the imbalance.
- If CV F1 ≈ test F1 (within ±0.02), the model generalises well and is not overfitting to the test split.
- AUC > 0.90 per class indicates the model is highly reliable in ranking users by risk — suitable for threshold-based intervention systems.

---

### 6.4 Key Findings

1. **`daily_usage_hours`, `avg_session_minutes`, and `screen_time_before_sleep`** are the three most important predictors of addiction level — all related to time intensity of usage.
2. **`mental_health_score`** is inversely correlated with addiction level — higher addiction is consistently associated with lower self-reported mental health.
3. **`night_usage`** adds meaningful signal beyond simple daily totals — when and how people use social media matters, not just how much.
4. **Platform and gender** are weak predictors — addiction risk is driven by usage behaviour, not demographic identity or platform choice.
5. **XGBoost outperforms** both Logistic Regression and Random Forest, confirming that the class boundaries are non-linear and benefit from gradient boosting's sequential error-correction.

---

### 6.5 Limitations & Future Improvements

- **Class imbalance:** The *High* class (16%) may benefit from SMOTE oversampling or `class_weight` tuning to improve recall for at-risk users.
- **Hyperparameter tuning:** Optuna or GridSearchCV on XGBoost `max_depth`, `subsample`, `colsample_bytree` could yield further gains.
- **Deployment:** The saved `addiction_model.pkl` can be wrapped in a FastAPI endpoint or Streamlit app for real-time prediction.
- **Explainability:** SHAP values would allow per-user explanations of why they were classified as High — essential for any intervention system.
- **Real data:** This synthetic dataset lacks temporal patterns (recency, frequency trends). Real-world telemetry data would allow more powerful time-series features.